1 try


based on normalized angles for each class 
This correction module uses:

Statistical modeling (per-class mean ± tolerance range).

Error normalization (z-score–like metric).

Human-interpretable mapping from deviations → linguistic feedback.

Algorithmically, this is equivalent to a multi-dimensional Gaussian deviation metric simplified for interpretability.


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path


# generatinge mean of each angle for each class

PROC_DIR = Path("../data/processed")
train_file = PROC_DIR / "processed_train.npz"
train_npz = np.load(train_file, allow_pickle=True)

angles = train_npz["angles"]   # shape (N, 9)
labels = train_npz["labels"]

templates = {}
for cls in np.unique(labels):
    cls_angles = angles[labels == cls]
    templates[cls] = np.mean(cls_angles, axis=0)

df_templates = pd.DataFrame(templates).T
df_templates.columns = [f"Angle_{i}" for i in range(df_templates.shape[1])]
df_templates.to_csv(PROC_DIR / "class_angle_templates.csv")


In [2]:
import numpy as np
import pandas as pd
import json
from pathlib import Path

# --- CONFIG ---
TOLERANCE_FACTOR = 1.5   # tolerance = 1.5 * std
MIN_TOL = 3.0            # minimum angle tolerance in degrees
E_MAX = 3.0              # normalization factor for overall correctness
ANGLE_NAMES = [
    "Left Elbow", "Right Elbow", "Left Shoulder", "Right Shoulder",
    "Left Hip", "Right Hip", "Left Knee", "Right Knee", "Torso Tilt"
]

# --- LOAD TEMPLATE ---
def load_templates(path="../data/processed/class_angle_templates.csv"):
    df = pd.read_csv(path, index_col=0)
    templates = {cls: df.loc[cls].values for cls in df.index}
    return templates

def load_std(path="../data/processed/class_angle_templates_std.csv"):
    if Path(path).exists():
        df = pd.read_csv(path, index_col=0)
        stds = {cls: df.loc[cls].values for cls in df.index}
    else:
        stds = None
    return stds

# --- SCORING FUNCTION ---
def score_pose(angles, cls, templates, stds=None):
    mu = templates[cls]
    if stds is not None:
        sigma = stds[cls]
    else:
        sigma = np.ones_like(mu) * 5.0  # fallback std if not saved

    tol = np.maximum(TOLERANCE_FACTOR * sigma, MIN_TOL)
    e = angles - mu
    s = np.abs(e) / tol  # normalized per-joint deviation
    s_weighted = s.mean()  # equal weights (for now)

    # aggregate correctness score
    S = max(0, 1 - min(s_weighted / E_MAX, 1))

    # detect incorrect joints
    incorrect = np.where(s > 1.0)[0]

    return {
        "score": float(S),
        "errors": e,
        "normalized_error": s,
        "incorrect_joints": incorrect
    }

# --- MESSAGE GENERATOR ---
def generate_feedback(result):
    msgs = []
    for idx in result["incorrect_joints"]:
        joint = ANGLE_NAMES[idx]
        delta = result["errors"][idx]
        deg = abs(delta)
        # choose intensity level
        if deg < 5: phrase = "slightly"
        elif deg < 15: phrase = "moderately"
        else: phrase = "significantly"

        direction = "straighten" if delta < 0 else "bend"
        msgs.append(f"{phrase} {direction} your {joint} ({deg:.1f}° off)")
    return msgs


In [4]:
# dummy usage try

import numpy as np

templates = load_templates()
stds = load_std()

sample_angles = np.array([150, 155, 75, 80, 105, 110, 165, 170, 10])  # sample frame
pose_class = "Akarna_Dhanurasana"  # example

result = score_pose(sample_angles, pose_class, templates, stds)
feedback = generate_feedback(result)

print("Pose Class:", pose_class)
print("Overall Correctness Score:", round(result["score"]*100, 2), "%")
print("Feedback Suggestions:")
for msg in feedback:
    print("-", msg)


Pose Class: Akarna_Dhanurasana
Overall Correctness Score: 0.0 %
Feedback Suggestions:
- significantly bend your Left Elbow (40.9° off)
- significantly bend your Right Elbow (53.0° off)
- significantly straighten your Left Shoulder (16.8° off)
- significantly bend your Left Hip (35.0° off)
- significantly bend your Right Hip (48.8° off)
- significantly bend your Left Knee (35.2° off)
- significantly bend your Right Knee (54.1° off)
- moderately straighten your Torso Tilt (10.7° off)


In [8]:
import numpy as np
import matplotlib.pyplot as plt
import mediapipe as mp
import cv2
from pathlib import Path
import pandas as pd

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

# Utility: draw landmarks and connections manually
def draw_skeleton(ax, landmarks, color=(0,1,0), title=None):
    connections = mp_pose.POSE_CONNECTIONS
    for connection in connections:
        start, end = connection
        p1 = landmarks[start]
        p2 = landmarks[end]
        ax.plot([p1[0], p2[0]], [-p1[1], -p2[1]], color=color, linewidth=2)
    ax.scatter(landmarks[:,0], -landmarks[:,1], c=[color], s=20)
    if title:
        ax.set_title(title)
    ax.axis('off')
    ax.set_aspect('equal')

# --- Load reference angles ---
PROC_DIR = Path("../data/processed")
ref_angles_df = pd.read_csv(PROC_DIR / "class_angle_templates.csv", index_col=0)

# Example: pick a pose class and load current frame
pose_class = "Akarna_Dhanurasana"
ref_angles = ref_angles_df.loc[pose_class].values

# Load one user's frame and extract landmarks
user_img_path = Path("../data/split/test/Akarna_Dhanurasana/0_54.jpg")
img = cv2.imread(str(user_img_path))



if img is None:
    print("Failed to load image")
else:
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
# img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

print(img_rgb)

with mp_pose.Pose(static_image_mode=True) as pose:
    results = pose.process(img_rgb)
    if results.pose_landmarks is None:
        print("No pose detected in image")
    else:
        user_lm = np.array([[l.x, l.y, l.z] for l in results.pose_landmarks.landmark])


# Normalize for consistent comparison
user_lm -= user_lm[24]  # shift origin to left hip
scale = np.linalg.norm(user_lm[12] - user_lm[11])  # shoulder width
user_lm /= scale

# Approximate reference skeleton by copying user skeleton, scaled to reference
# (You can later replace this with precomputed mean landmarks)
ref_lm = user_lm.copy()

# Compute difference color map per joint (for demonstration)
error_magnitude = np.abs(ref_angles - np.random.normal(ref_angles, 5, len(ref_angles)))  # fake difference demo
color_map = plt.cm.RdYlGn(1 - (error_magnitude / error_magnitude.max()))

# --- Plot ---
fig, axes = plt.subplots(1,2, figsize=(12,6))
draw_skeleton(axes[0], ref_lm, color=(0,0.7,0), title=f"Ideal Skeleton: {pose_class}")
draw_skeleton(axes[1], user_lm, color=(1,0,0), title="User's Current Pose")

plt.suptitle("Comparison: Reference vs User Pose", fontsize=14)
plt.tight_layout()
plt.show()


[[[255 255 255]
  [255 255 255]
  [255 255 255]
  ...
  [255 255 255]
  [255 255 255]
  [255 255 255]]

 [[255 255 255]
  [255 255 255]
  [255 255 255]
  ...
  [255 255 255]
  [255 255 255]
  [255 255 255]]

 [[255 255 255]
  [255 255 255]
  [255 255 255]
  ...
  [255 255 255]
  [255 255 255]
  [255 255 255]]

 ...

 [[255 255 255]
  [255 255 255]
  [255 255 255]
  ...
  [255 255 255]
  [255 255 255]
  [255 255 255]]

 [[255 255 255]
  [255 255 255]
  [255 255 255]
  ...
  [255 255 255]
  [255 255 255]
  [255 255 255]]

 [[255 255 255]
  [255 255 255]
  [255 255 255]
  ...
  [255 255 255]
  [255 255 255]
  [255 255 255]]]
No pose detected in image


NameError: name 'user_lm' is not defined